In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
import pandas as pd
import numpy as np
import random
import pickle
import re
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime
from tqdm import tqdm
from sklearn.metrics import r2_score
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split

from data.data_loader_extend_for_lstm import EPCDataset, UMassDataset
from models.utils_extend import create_model, train_for_long_term_forecast, train_for_short_term_forecast, evaluate_for_long_term_forecast, evaluate_for_short_term_forecast

from explainers.utils_extend import get_explainer, unpack_eval_for_single

## Feature 추출

In [2]:
is_long_term_forecast = True

lc_output_file = "results/umass_important_features_for_lf.txt"
sc_output_file = "results/umass_important_features_for_sf.txt"

umass_path = 'data/umass/HomeA/HomeA_with_weather.csv'

def get_base_features(path):
    df = pd.read_csv(path)
    features = df.columns.tolist()
    return [f.lower() for f in features]

base_features = get_base_features(umass_path)
len(base_features)

48

In [3]:
model_path_pattern = re.compile(r"\./trained_models/(.+)")
feature_pattern = re.compile(r"([\w_]+): ([\d\.]+)")

important_features_dict = {}
current_model = None
current_section = None

if is_long_term_forecast:
    with open(lc_output_file, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            
            # 모델 경로 감지 및 모델 키 추출
            match = model_path_pattern.match(line)
            if match:
                current_model = match.group(1)
                important_features_dict[current_model] = {"long": {}, "short": {}}
                continue

            # 섹션 감지
            if "Important Long-term Features" in line:
                current_section = "long"
                continue
            elif "Important Short-term Features" in line:
                current_section = "short"
                continue
            
            # Feature 값 추출
            match = feature_pattern.match(line)
            if match and current_model and current_section:
                feature_name, score = match.groups()
                if feature_name in base_features:
                    important_features_dict[current_model][current_section][feature_name] = score

else:
    with open(sc_output_file, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            
            match = model_path_pattern.match(line)
            if match:
                current_model = match.group(1)
                important_features_dict[current_model] = []
                continue

            # Feature 값 추출
            match = feature_pattern.match(line)
            if match and current_model:
                feature_name, score = match.groups()
                if feature_name in base_features:
                    important_features_dict[current_model].append(feature_name)

In [4]:
important_features_dict

{'LS_CNNLSTM_umass_long2160_short720_pred256_hs256_nl3_a0.3_b1.0.pth': {'long': {'masterbathoutlets_m3': '2.4884146504928264',
   'dishwasherdisposalsinklight_m4': '2.452413633210933',
   'rearbasementlights_m4': '2.432417037984603',
   'denoutdoorlights_m4': '2.3901889000292784',
   'garageoutlets_m3': '2.375501203039082',
   'mudroomoutlets_m3': '2.369883764410498',
   'denoutlets_m4': '2.3570400370331237',
   'masterbedbathlights_m4': '2.310792554326214',
   'kitchenoutletseast_m4': '2.3033520579901965',
   'officelights_m4': '2.2887151070306166',
   'use_m4': '2.254853881120294',
   'kitchendenlights_m4': '2.2527411033573626',
   'basementoutdooroutlets_m3': '2.2433424941209408',
   'kitchenoutletssouth_m4': '2.144247436768523',
   'microwave_m4': '2.110978714030827',
   'diningroomoutlets_m3': '2.10085027868009',
   'masteroutlets_m4': '2.0337003831793115',
   'refrigerator_m4': '1.9961991620435855',
   'pressure': '1.2679644421449896',
   'use_total': '1.2278323862896463',
   'dr

### Top-K

In [5]:
# 대략 20–60% 축소 (20%, 30%, 40%, 50%, 60%)

num_base_features = len(base_features)
extracted_features = {}
top_k_th = [0.15, 0.3, 0.45, 0.6, 0.75]

for i in reversed(range(1,6)):
    key_name = 'top_{}'.format(i)
    print('key_name: ', key_name)

    top_k_features = {}
    top_k_rate = top_k_th[i-1]
    
    for model_name, all_features in important_features_dict.items():
        top_k_features[model_name] = {}
        num_top_k_features = int(num_base_features*top_k_rate)

        long_feature_name = list(all_features['long'].keys())
        top_k_long_features = long_feature_name[:num_top_k_features]
        top_k_features[model_name]['long'] = top_k_long_features

        short_feature_name = list(all_features['short'].keys())
        top_k_short_features = short_feature_name[:num_top_k_features]
        top_k_features[model_name]['short'] = top_k_short_features
        
        print('top_k_features[model_name][long]: ', len(top_k_features[model_name]['long']))

    extracted_features[key_name] = top_k_features

key_name:  top_5
top_k_features[model_name][long]:  36
top_k_features[model_name][long]:  36
top_k_features[model_name][long]:  36
top_k_features[model_name][long]:  36
key_name:  top_4
top_k_features[model_name][long]:  28
top_k_features[model_name][long]:  28
top_k_features[model_name][long]:  28
top_k_features[model_name][long]:  28
key_name:  top_3
top_k_features[model_name][long]:  21
top_k_features[model_name][long]:  21
top_k_features[model_name][long]:  21
top_k_features[model_name][long]:  21
key_name:  top_2
top_k_features[model_name][long]:  14
top_k_features[model_name][long]:  14
top_k_features[model_name][long]:  14
top_k_features[model_name][long]:  14
key_name:  top_1
top_k_features[model_name][long]:  7
top_k_features[model_name][long]:  7
top_k_features[model_name][long]:  7
top_k_features[model_name][long]:  7


### method1~4

In [6]:
### method 1: long, short 각각 중요도 점수 비율 
def select_features_by_termwise_ratio(scores, threshold_ratio=0.7):
    sorted_feats = sorted(scores.items(), key=lambda x: float(x[1]), reverse=True)
    total = sum(float(v) for _, v in sorted_feats)
    selected = []
    running_sum = 0
    for f, v in sorted_feats:
        running_sum += float(v)
        selected.append(f)
        if running_sum / total >= threshold_ratio:
            break
    return selected


### method 2: long, short 중요도 합의 점수 비율
def select_features_by_combined_score(long_scores, short_scores, threshold_ratio=0.7):
    all_features = set(long_scores) | set(short_scores)
    combined_scores = {
        f: float(long_scores.get(f, 0)) + float(short_scores.get(f, 0))
        for f in all_features
    }
    sorted_feats = sorted(combined_scores.items(), key=lambda x: float(x[1]), reverse=True)
    total = sum(score for _, score in sorted_feats)
    selected = []
    running_sum = 0
    for f, score in sorted_feats:
        running_sum += score
        selected.append(f)
        if running_sum / total >= threshold_ratio:
            break
    return selected


### method 3: long, short 중요도 순위 합
def select_by_rank_sum(long_ranked: list, short_ranked: list, max_rank_sum: int = 40):
    selected = []
    for feat in long_ranked:
        if feat in short_ranked:
            long_rank = long_ranked.index(feat) + 1  # 순위는 1부터
            short_rank = short_ranked.index(feat) + 1
            if long_rank + short_rank <= max_rank_sum:
                selected.append(feat)
    return selected


### method 4: long, short 중요도 순위 상대 거리
def select_by_relative_rank_distance(
    base_ranked: list,  # 중심 기준 (long or short)
    reference_ranked: list,  # 상대 비교 term
    top_k: int = 40,
    max_distance: int = 6
):
    selected = []
    for i, feat in enumerate(base_ranked[:top_k]):  # 기준 term에서 상위 top_k만
        if feat in reference_ranked:
            ref_rank = reference_ranked.index(feat)
            if abs(ref_rank - i) <= max_distance:
                selected.append(feat)
    return selected

In [7]:
filtered_features_1 = {}
filtered_features_2 = {}
filtered_features_3 = {}
filtered_features_4 = {}

for model_name, all_features in important_features_dict.items():
    filtered_features_1[model_name] = {}
    long_selected = select_features_by_termwise_ratio(all_features['long'])
    short_selected = select_features_by_termwise_ratio(all_features['short'])
    filtered_features_1[model_name]['long'] = long_selected
    filtered_features_1[model_name]['short'] = short_selected
    # print(len(filtered_features_1[model_name]['long']))
    # print(len(filtered_features_1[model_name]['short']), '\n')

    ### method 2
    filtered_features_2[model_name] = {}
    selected_feastures = select_features_by_combined_score(all_features['long'], all_features['short'])
    filtered_features_2[model_name]['long'] = selected_feastures
    filtered_features_2[model_name]['short'] = selected_feastures
    # print(len(filtered_features_2[model_name]['long']))
    # print(len(filtered_features_2[model_name]['short']), '\n')

    ### method 3
    filtered_features_3[model_name] = {}
    selected_feastures = select_by_rank_sum(list(all_features['long']), list(all_features['short']))
    filtered_features_3[model_name]['long'] = selected_feastures
    filtered_features_3[model_name]['short'] = selected_feastures
    # print(len(filtered_features_3[model_name]['long']))
    # print(len(filtered_features_3[model_name]['short']), '\n')

    ### method 4
    filtered_features_4[model_name] = {}
    long_selected = select_by_relative_rank_distance(list(all_features['long']), list(all_features['short']))
    short_selected = select_by_relative_rank_distance(list(all_features['short']), list(all_features['long']))
    filtered_features_4[model_name]['long'] = long_selected
    filtered_features_4[model_name]['short'] = short_selected
    # print(len(filtered_features_4[model_name]['long']))
    # print(len(filtered_features_4[model_name]['short']), '\n')

In [8]:
extracted_features['method1'] = filtered_features_1
extracted_features['method2'] = filtered_features_2
extracted_features['method3'] = filtered_features_3
extracted_features['method4'] = filtered_features_4

In [9]:
extracted_features

{'top_5': {'LS_CNNLSTM_umass_long2160_short720_pred256_hs256_nl3_a0.3_b1.0.pth': {'long': ['masterbathoutlets_m3',
    'dishwasherdisposalsinklight_m4',
    'rearbasementlights_m4',
    'denoutdoorlights_m4',
    'garageoutlets_m3',
    'mudroomoutlets_m3',
    'denoutlets_m4',
    'masterbedbathlights_m4',
    'kitchenoutletseast_m4',
    'officelights_m4',
    'use_m4',
    'kitchendenlights_m4',
    'basementoutdooroutlets_m3',
    'kitchenoutletssouth_m4',
    'microwave_m4',
    'diningroomoutlets_m3',
    'masteroutlets_m4',
    'refrigerator_m4',
    'pressure',
    'use_total',
    'dryer_m3',
    'furnacehrv_m2',
    'bedroomoutlets_m2',
    'temperature',
    'masteroutlets_m2',
    'humidity',
    'windspeed',
    'disposaldishwasher_m2',
    'electricrange_m3',
    'use_m2',
    'garagemudroomlights_m3',
    'use_m3',
    'apparenttemperature',
    'fridgerange_m2',
    'kitchenlights_m2',
    'cellaroutlets_m2'],
   'short': ['officelights_m4',
    'kitchenoutletseast_m4',

## Data load

In [10]:
dataset_params = {
    # Long-term forecast
    'long_term_length' : 90*24,           # 입력 길이 (예: 90일치, 1시간 단위)
    'long_term_pred_length' : 256,        # 출력 길이 (예: 256시간)

    # Short-term forecast
    'short_term_length' : 30*24,          # 입력 길이 (예: 30일치)
    'short_term_pred_length' : 7*24,      # 출력 길이 (예: 7일치)

    # 단일 short-term (LSTM/GRU/CNN-LSTM 전용)
    'sequence_length' : 24*30,            # 입력 길이 (예: 30일치)
    'prediction_length' : 24              # 출력 길이 (예: 하루치)
}

dataset_name = 'umass'

In [11]:
def _resolve_filtered_list(filtered_features, which):
    """
    filtered_features가 dict({'long': [...], 'short': [...]}) 또는 list([...]) 또는 None일 수 있다.
    which: 'long' | 'short' | 'single'
    반환: list 또는 None
    """
    if filtered_features is None:
        return None
    if isinstance(filtered_features, dict):
        # dict이면 해당 키가 있으면 그걸 사용, 없으면 None
        return filtered_features.get(which)
    # list면 그대로 사용
    if isinstance(filtered_features, (list, tuple)):
        return list(filtered_features)
    # 예외적으로 문자열 등 들어오면 None 처리
    return None

In [12]:
def load_dataset(params,
                 dataset_type="epc",
                 file_path=None,
                 is_long_term_forecast=True,
                 target_name=None,
                 categorical_features=('icon',),
                 filtered_features=None):
    """
    params: {
        # long/short 모두 쓰는 경우
        "long_term_length": int,
        "long_term_pred_length": int,
        "short_term_length": int,
        "short_term_pred_length": int,

        # single 모드만 쓰는 경우
        "sequence_length": int,
        "prediction_length": int,
    }

    dataset_type: "epc" | "umass"
    target_name:  타깃 컬럼명 하나만 명시(Optional). None이면
                  EPC→Global_active_power, UMass→use_total, 없으면 첫 숫자형.
    categorical_features: UMass에서 임베딩 인덱스로 뽑을 범주형 컬럼 tuple
    """
    selected_features = {}

    # 어떤 클래스를 쓸지 선택
    if dataset_type.lower() == "epc":
        DatasetClass = EPCDataset
        is_umass = False
        ds_extra_kwargs = {}
    elif dataset_type.lower() == "umass":
        DatasetClass = UMassDataset
        is_umass = True
        # 여기서는 공통 kwargs만 넣고, filtered_features는 인스턴스 생성 시점에 주입
        ds_extra_kwargs = {
            "categorical_features": categorical_features,
        }
    else:
        raise ValueError(f"Unknown dataset_type: {dataset_type}")

    if is_long_term_forecast:
        # ----- Long -----
        ds_long = DatasetClass(
            file_path=file_path,
            sequence_length=params["long_term_length"],
            prediction_length=params["long_term_pred_length"],
            target_name=target_name,
            filtered_features=_resolve_filtered_list(filtered_features, "long"), 
            **ds_extra_kwargs
        )
        
        long_out = ds_long.load_data()
        target_scaler_long = ds_long.target_scaler
        
        # ----- Short -----
        ds_short = DatasetClass(
            file_path=file_path,
            sequence_length=params["short_term_length"],
            prediction_length=params["short_term_pred_length"],
            target_name=target_name,
            filtered_features=_resolve_filtered_list(filtered_features, "short"), 
            **ds_extra_kwargs
        )
        short_out = ds_short.load_data()
        target_scaler_short = ds_short.target_scaler

        target_scaler = (target_scaler_long, target_scaler_short)

        if is_umass:
            # UMass: 6개 반환
            Xn_tr_L, Xi_tr_L, y_tr_L, Xn_ev_L, Xi_ev_L, y_ev_L = long_out
            Xn_tr_S, Xi_tr_S, y_tr_S, Xn_ev_S, Xi_ev_S, y_ev_S = short_out

            train_data    = ((Xn_tr_L, Xi_tr_L), (Xn_tr_S, Xi_tr_S))
            train_targets = (y_tr_L, y_tr_S)
            eval_data     = ((Xn_ev_L, Xi_ev_L), (Xn_ev_S, Xi_ev_S))
            eval_targets  = (y_ev_L, y_ev_S)
        else:
            # EPC: 4개 반환
            train_long, train_targets_long, eval_long, eval_targets_long = long_out
            train_short, train_targets_short, eval_short, eval_targets_short = short_out

            train_data    = (train_long,  train_short)
            train_targets = (train_targets_long, train_targets_short)
            eval_data     = (eval_long,   eval_short)
            eval_targets  = (eval_targets_long, eval_targets_short)

        # auto-selected numeric feature names를 우선 제공
        get_names = lambda ds: (ds.get_numeric_feature_names()
                                if hasattr(ds, "get_numeric_feature_names")
                                else getattr(ds, "selected_features", []))
        selected_features["long"]  = get_names(ds_long)
        selected_features["short"] = get_names(ds_short)

    else:
        # ----- Single (short only) -----
        ds_short = DatasetClass(
            file_path=file_path,
            sequence_length=params["sequence_length"],
            prediction_length=params["prediction_length"],
            target_name=target_name,
            filtered_features=_resolve_filtered_list(filtered_features, "single"),  # 👈 추가
            **ds_extra_kwargs
        )
        out = ds_short.load_data()
        target_scaler = ds_short.target_scaler

        if is_umass:
            Xn_tr, Xi_tr, y_tr, Xn_ev, Xi_ev, y_ev = out
            train_data    = (Xn_tr, Xi_tr)
            train_targets = y_tr
            eval_data     = (Xn_ev, Xi_ev)
            eval_targets  = y_ev
        else:
            train_seq, train_tgt, eval_seq, eval_tgt = out
            train_data,  train_targets = train_seq, train_tgt
            eval_data,   eval_targets  = eval_seq,  eval_tgt

        if hasattr(ds_short, "get_numeric_feature_names"):
            selected_features["single"] = ds_short.get_numeric_feature_names()
        else:
            selected_features["single"] = getattr(ds_short, "selected_features", [])

    return train_data, train_targets, eval_data, eval_targets, selected_features, target_scaler


In [13]:
def new_load_trained_model(
    params,
    fs_name,
    is_long_term_forecast=True,
    oversample_eval=True,
    icon_embed_dim=8,
    path_suffix=None
):
    if params is None:
        raise ValueError("params must be provided")

    model_name    = params['model_name']
    hidden_size   = params['hidden_size']
    num_layers    = params['num_layers']
    dropout       = params['dropout']
    num_epochs    = params['num_epochs']
    batch_size    = params['batch_size']
    learning_rate = params['learning_rate']
    patience      = params['patience']
    mse_decay     = params.get('mse_decay', False)

    output_size = {
        'long'  : dataset_params.get('long_term_pred_length', dataset_params.get('prediction_length', 24)),
        'short' : dataset_params.get('short_term_pred_length', dataset_params.get('prediction_length', 24)),
        'single': dataset_params.get('prediction_length', 24),
    }

    mse_alpha, mse_beta = (0.3, 1.0) if mse_decay else (1.0, 1.0)

    icon_vocab_size = None

    if is_long_term_forecast:
        # EPC: train_data = (XL, XS)
        # UMass: train_data = ((XnL, XiL), (XnS, XiS))
        train_long, train_short = train_data
        is_umass = isinstance(train_long, (tuple, list)) and isinstance(train_short, (tuple, list))
    
        if is_umass:
            XnL, XiL = train_long
            XnS, XiS = train_short
            in_long  = XnL.shape[2]
            in_short = XnS.shape[2]
            icon_vocab_size = int(torch.max(torch.stack([XiL.max(), XiS.max()])).item()) + 1
        else:
            XL, XS = train_long, train_short
            in_long  = XL.shape[2]
            in_short = XS.shape[2]
            icon_vocab_size = None  # EPC
    
        long_output_size  = dataset_params['long_term_pred_length']
        short_output_size = dataset_params['short_term_pred_length']
    
        model = create_model(
            model_name=model_name,
            # 장·단기 입력/출력
            long_input_size_num=in_long,
            short_input_size_num=in_short,
            long_output_size=long_output_size,
            short_output_size=short_output_size,
            # 공통
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            # 아이콘(UMass만 유효)
            icon_vocab_size=icon_vocab_size,
            icon_embed_dim=icon_embed_dim,
            # 참고: 길이(모델이 필요로 하면)
            long_term_length=dataset_params['long_term_length'],
            short_term_length=dataset_params['short_term_length']
        )

        suffix = path_suffix or ("umass" if icon_vocab_size else "epc")
        model_path = './trained_models/{}_{}_{}_long{}_short{}_pred{}_hs{}_nl{}_a{}_b{}.pth'.format(
            fs_name,
            model_name, suffix,
            dataset_params['long_term_length'], dataset_params['short_term_length'],
            dataset_params['long_term_pred_length'],
            hidden_size, num_layers, mse_alpha, mse_beta
        )
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        print("Model path:", model_path)

        # ---------------------------
        # 로드 or 학습
        # ---------------------------
        new_saved = True
        if os.path.exists(model_path):
            new_saved = False
            print(f"Loading the pre-trained {model_name} ...")
            model.load_state_dict(torch.load(model_path, map_location='cpu'))
            model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
        else:
            print(f"{model_name} not found. Training a new one (long-term)...")
            train_for_long_term_forecast(
                model=model,
                model_name=model_name,
                train_data=train_data,          # 전역 (EPC: (XL, XS) / UMass: ((XnL, XiL), (XnS, XiS)))
                train_targets=train_targets,    # (yL, yS)
                eval_data=eval_data,            # 구조 동일
                eval_targets=eval_targets,      # (yL, yS)
                model_path=model_path,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                patience=patience,
                oversample_eval=oversample_eval,  # 평가시 오버샘플 여부
                alpha=mse_alpha,
                beta=mse_beta
            )

    else:
        # EPC: train_data = Xn
        # UMass: train_data = (Xn, Xi)
        is_umass = isinstance(train_data, (tuple, list))
    
        if is_umass:
            Xn, Xi = train_data
            in_single = Xn.shape[2]
            icon_vocab_size = int(Xi.max().item()) + 1
        else:
            Xn = train_data
            in_single = Xn.shape[2]
            icon_vocab_size = None
    
        output_size_single = dataset_params.get('prediction_length', 24)
    
        model = create_model(
            model_name=model_name,
            input_size_num=in_single,
            output_size=output_size_single,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            icon_vocab_size=icon_vocab_size,
            icon_embed_dim=icon_embed_dim
        )

        suffix = path_suffix or ("umass" if icon_vocab_size else "epc")
        model_path = './trained_models/{}_{}_{}_seq{}_pred{}_hs{}_nl{}.pth'.format(
            fs_name,
            model_name, suffix,
            dataset_params['sequence_length'], dataset_params['prediction_length'],
            hidden_size, num_layers
        )
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        print("Model path:", model_path)
    
        if os.path.exists(model_path):
            print(f"Loading the pre-trained {model_name} ...")
            model.load_state_dict(torch.load(model_path, map_location='cpu'))
            model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
        else:
            print(f"{model_name} not found. Training a new one...")
            train_for_short_term_forecast(
                model=model,
                model_name=model_name,
                train_sequences=train_data,
                train_targets=train_targets,
                eval_sequences=eval_data,
                eval_targets=eval_targets,
                model_path=model_path,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                patience=patience
            )

    return model, model_path, new_saved

In [16]:
result_path = 'results/long_topk_methods.txt'

for selection_method, all_features in extracted_features.items():
    print('selection_method: ', selection_method)

    for model_full_name, features in all_features.items():
        print('model_full_name: ', model_full_name)

        long_features = []
        short_features = []
        
        for f in features['long']:
            long_features.append(f)
        for f in features['short']:
            short_features.append(f)

        train_data, train_targets, eval_data, eval_targets, selected_features, target_scaler = load_dataset(
            params=dataset_params,
            dataset_type=dataset_name,
            file_path="data/umass/HomeA/HomeA_with_weather.csv",
            is_long_term_forecast=is_long_term_forecast,
            target_name="use_total",
            categorical_features=('icon',),
            filtered_features={'long':long_features, 'short':short_features}
        )

        model_name = model_full_name.split('_umass')[0]
        if 'Att' in model_name:
            continue 
            
        hidden_size = model_full_name.split('hs')[1].split('_')[0]

        model_params = {
            'model_name'  : model_name,
            'hidden_size' : int(hidden_size),
            'num_layers'  : 3,
            'dropout'     : 0.3,
            'num_epochs'  : 200,
            'batch_size'  : 128,
            'learning_rate': 0.001,
            'patience'    : 15,
            'mse_decay'   : True
        }
        
        if 'model' in locals():
            del model  # 기존 모델 삭제
            torch.cuda.empty_cache()  # GPU 메모리 해제
                    
        model, model_path, new_saved = new_load_trained_model(
            params=model_params,
            fs_name=selection_method,
            is_long_term_forecast=is_long_term_forecast,
            oversample_eval=True  # 평가셋 오버샘플 여부(기존 정책)
        )

        if new_saved: 
            print("Evaluating the model...")
            if is_long_term_forecast:
                results = evaluate_for_long_term_forecast(
                    model=model,
                    eval_data=eval_data,
                    eval_targets=eval_targets,
                    model_name=model_params['model_name'],
                    batch_size=model_params['batch_size'],
                    oversample_eval=True,     # 기존 정책 유지
                    # target_scaler=target_scaler
                )
            else:
                results = evaluate_for_short_term_forecast(
                    model=model,
                    eval_sequences=eval_data,     # EPC는 X, UMass는 (Xn, Xi)
                    eval_targets=eval_targets,
                    model_name=model_params['model_name'],
                    batch_size=model_params['batch_size'],
                    # target_scaler=target_scaler
                )

            with open(result_path, "a") as f:
                f.write('{}\n'.format(model_path))
                f.write('{}\n\n\n'.format(results))


        del model
        torch.cuda.empty_cache()

selection_method:  top_5
model_full_name:  LS_CNNLSTM_umass_long2160_short720_pred256_hs256_nl3_a0.3_b1.0.pth
Model path: ./trained_models/top_5_LS_CNNLSTM_umass_long2160_short720_pred256_hs256_nl3_a0.3_b1.0.pth
Loading the pre-trained LS_CNNLSTM ...
model_full_name:  LS_CNNLSTM_umass_long2160_short720_pred256_hs512_nl3_a0.3_b1.0.pth


/tmp/ipykernel_1170049/797558504.py:90: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location='cpu'))


Model path: ./trained_models/top_5_LS_CNNLSTM_umass_long2160_short720_pred256_hs512_nl3_a0.3_b1.0.pth
Loading the pre-trained LS_CNNLSTM ...
model_full_name:  LS_CNNLSTM_Att_umass_long2160_short720_pred256_hs256_nl3_a0.3_b1.0.pth


/tmp/ipykernel_1170049/797558504.py:90: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location='cpu'))


model_full_name:  LS_CNNLSTM_Att_umass_long2160_short720_pred256_hs512_nl3_a0.3_b1.0.pth
selection_method:  top_4
model_full_name:  LS_CNNLSTM_umass_long2160_short720_pred256_hs256_nl3_a0.3_b1.0.pth
Model path: ./trained_models/top_4_LS_CNNLSTM_umass_long2160_short720_pred256_hs256_nl3_a0.3_b1.0.pth
Loading the pre-trained LS_CNNLSTM ...
model_full_name:  LS_CNNLSTM_umass_long2160_short720_pred256_hs512_nl3_a0.3_b1.0.pth


...

model_full_name:  LS_CNNLSTM_Att_umass_long2160_short720_pred256_hs512_nl3_a0.3_b1.0.pth
selection_method:  method4
model_full_name:  LS_CNNLSTM_umass_long2160_short720_pred256_hs256_nl3_a0.3_b1.0.pth
Model path: ./trained_models/method4_LS_CNNLSTM_umass_long2160_short720_pred256_hs256_nl3_a0.3_b1.0.pth
Loading the pre-trained LS_CNNLSTM ...
model_full_name:  LS_CNNLSTM_umass_long2160_short720_pred256_hs512_nl3_a0.3_b1.0.pth


/tmp/ipykernel_1170049/797558504.py:90: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location='cpu'))


Model path: ./trained_models/method4_LS_CNNLSTM_umass_long2160_short720_pred256_hs512_nl3_a0.3_b1.0.pth
Loading the pre-trained LS_CNNLSTM ...
model_full_name:  LS_CNNLSTM_Att_umass_long2160_short720_pred256_hs256_nl3_a0.3_b1.0.pth


/tmp/ipykernel_1170049/797558504.py:90: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location='cpu'))


model_full_name:  LS_CNNLSTM_Att_umass_long2160_short720_pred256_hs512_nl3_a0.3_b1.0.pth


In [16]:
# target_fs = 'top_5'
# target_model = 'LS_CNNLSTM'

# for selection_method, all_features in extracted_features.items():
#     print('selection_method: ', selection_method)
#     if not(target_fs == selection_method):
#         print('skip fs')
#         continue 

#     for model_full_name, features in all_features.items():
#         print('model_full_name: ', model_full_name)
#         if not(target_model == model_full_name.split('_umass')[0]):
#             print('skip model')
#             continue 

#         long_features = []
#         short_features = []
        
#         for f in features['long']:
#             long_features.append(f)
#         for f in features['short']:
#             short_features.append(f)

#         train_data, train_targets, eval_data, eval_targets, selected_features, target_scaler = load_dataset(
#             params=dataset_params,
#             dataset_type=dataset_name,
#             file_path="data/umass/HomeA/HomeA_with_weather.csv",
#             is_long_term_forecast=is_long_term_forecast,
#             target_name="use_total",
#             categorical_features=('icon',),
#             filtered_features={'long':long_features, 'short':short_features}
#         )

#         model_name = model_full_name.split('_umass')[0]
#         hidden_size = model_full_name.split('hs')[1].split('_')[0]

#         model_params = {
#             'model_name'  : model_name,
#             'hidden_size' : int(hidden_size),
#             'num_layers'  : 3,
#             'dropout'     : 0.3,
#             'num_epochs'  : 200,
#             'batch_size'  : 128,
#             'learning_rate': 0.001,
#             'patience'    : 15,
#             'mse_decay'   : True
#         }
        
#         model, model_path, new_saved = new_load_trained_model(
#             params=model_params,
#             fs_name=selection_method,
#             is_long_term_forecast=is_long_term_forecast,
#             oversample_eval=True  # 평가셋 오버샘플 여부(기존 정책)
#         )

#         print("Evaluating the model...")
#         if is_long_term_forecast:
#             results = evaluate_for_long_term_forecast(
#                 model=model,
#                 eval_data=eval_data,
#                 eval_targets=eval_targets,
#                 model_name=model_params['model_name'],
#                 batch_size=model_params['batch_size'],
#                 oversample_eval=True,     # 기존 정책 유지
#                 # target_scaler=target_scaler
#             )
#         else:
#             results = evaluate_for_short_term_forecast(
#                 model=model,
#                 eval_sequences=eval_data,     # EPC는 X, UMass는 (Xn, Xi)
#                 eval_targets=eval_targets,
#                 model_name=model_params['model_name'],
#                 batch_size=model_params['batch_size'],
#                 # target_scaler=target_scaler
#             )

selection_method:  top_5
model_full_name:  LS_CNNLSTM_umass_long2160_short720_pred256_hs256_nl3_a0.3_b1.0.pth
Model path: ./trained_models/top_5_LS_CNNLSTM_umass_long2160_short720_pred256_hs256_nl3_a0.3_b1.0.pth
Loading the pre-trained LS_CNNLSTM ...


/tmp/ipykernel_1201643/797558504.py:90: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location='cpu'))


Evaluating the model...
[Short] R²: 0.6045, Adj.R²: 0.6012, SMAPE: 28.57, MASE: 1.3704
[Long ] R²: 0.5999, Adj.R²: 0.5966, SMAPE: 30.25, MASE: 1.3348
model_full_name:  LS_CNNLSTM_umass_long2160_short720_pred256_hs512_nl3_a0.3_b1.0.pth
Model path: ./trained_models/top_5_LS_CNNLSTM_umass_long2160_short720_pred256_hs512_nl3_a0.3_b1.0.pth
Loading the pre-trained LS_CNNLSTM ...


/tmp/ipykernel_1201643/797558504.py:90: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location='cpu'))


Evaluating the model...
[Short] R²: 0.8540, Adj.R²: 0.8527, SMAPE: 22.14, MASE: 0.7808
[Long ] R²: 0.8508, Adj.R²: 0.8496, SMAPE: 23.09, MASE: 0.8008
model_full_name:  LS_CNNLSTM_Att_umass_long2160_short720_pred256_hs256_nl3_a0.3_b1.0.pth
skip model
model_full_name:  LS_CNNLSTM_Att_umass_long2160_short720_pred256_hs512_nl3_a0.3_b1.0.pth
skip model
selection_method:  top_4
skip fs
selection_method:  top_3
skip fs
selection_method:  top_2
skip fs
selection_method:  top_1
skip fs
selection_method:  method1
skip fs
selection_method:  method2
skip fs
selection_method:  method3
skip fs
selection_method:  method4
skip fs
